
# Product Price Prediction. Fine-Tuning GPT-4.1-nano
A fine-tuned LLM that estimates Amazon product prices from descriptions. Play "The Price is Right": guess the price, then see how you stack up
against the base model and the fine-tuned model side by side.

In [2]:
# imports

import os
import re
import json
import random
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
import gradio as gr
import sys
sys.path.append('../../')
from pricer.items import Item
from pricer.evaluator import evaluate

In [3]:
# constants

BASE_MODEL   = "gpt-4.1-nano-2025-04-14"
TRAIN_SIZE   = 200
VAL_SIZE     = 50
JSONL_FOLDER = "jsonl"

SYSTEM_PROMPT = (
    "You are a product pricing assistant with deep knowledge of the Amazon marketplace. "
    "Given a product title, category, brand, and description, estimate its price in USD. "
    "Respond with the price only, in the format: Price is $X.XX. No explanation."
)

In [4]:
# set up environment
load_dotenv(override=True)

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)
    print("✅ HuggingFace token found")
else:
    print("❌ HuggingFace token not found")

openai_api_key = os.environ.get("OPENAI_API_KEY")
if openai_api_key:
    print("✅ OpenAI API key found")
else:
    print("❌ OpenAI API key not found")

openai = OpenAI()


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HuggingFace token found
✅ OpenAI API key found


In [5]:
# load dataset

train, val, test = Item.from_hub("ed-donner/items_lite")
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

fine_tune_train = train[:TRAIN_SIZE]
fine_tune_val   = val[:VAL_SIZE]


README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00000-of-00001.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


validation-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [6]:
# prepare training data

def messages_for(item):
    return [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Estimate the price of this product.\n\n{item.summary}"},
        {"role": "assistant", "content": f"Price is ${item.price:.2f}"},
    ]

def make_jsonl(items):
    result = ""
    for item in items:
        messages     = messages_for(item)
        messages_str = json.dumps(messages)
        result      += '{"messages": ' + messages_str + '}\n'
    return result.strip()

def write_jsonl(items, filename):
    Path(JSONL_FOLDER).mkdir(exist_ok=True)
    with open(filename, "w") as f:
        f.write(make_jsonl(items))

write_jsonl(fine_tune_train, f"{JSONL_FOLDER}/fine_tune_train.jsonl")
write_jsonl(fine_tune_val,   f"{JSONL_FOLDER}/fine_tune_validation.jsonl")
print("✅ JSONL files written")


print(json.dumps({"messages": messages_for(fine_tune_train[0])}, indent=2))

✅ JSONL files written
{
  "messages": [
    {
      "role": "system",
      "content": "You are a product pricing assistant with deep knowledge of the Amazon marketplace. Given a product title, category, brand, and description, estimate its price in USD. Respond with the price only, in the format: Price is $X.XX. No explanation."
    },
    {
      "role": "user",
      "content": "Estimate the price of this product.\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."
    },
    {
      "role": "assistant",
      "content": "Price is $64.30"
    }
  ]
}


In [7]:
# upload to OpenAI

with open(f"{JSONL_FOLDER}/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
print(f"✅ Train file uploaded: {train_file.id}")

with open(f"{JSONL_FOLDER}/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")
print(f"✅ Validation file uploaded: {validation_file.id}")

✅ Train file uploaded: file-3DsE8HEksXYCjFYTa5R4A2
✅ Validation file uploaded: file-YRZRZUq5wLwwpUfdMkU2Bp


In [8]:
# submit fine-tuning job

job = openai.fine_tuning.jobs.create(
    training_file   = train_file.id,
    validation_file = validation_file.id,
    model           = BASE_MODEL,
    seed            = 42,
    hyperparameters = {"n_epochs": 1, "batch_size": 1},
    suffix          = "pricer"
)
print(f"✅ Fine-tuning job submitted: {job.id}")
print("Track progress at: https://platform.openai.com/finetune")

✅ Fine-tuning job submitted: ftjob-OuBMK4JJhpALvfHlgKX2tVul
Track progress at: https://platform.openai.com/finetune


In [15]:
# Monitor Job
# Re-run this cell until status shows 'succeeded'
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
status = openai.fine_tuning.jobs.retrieve(job_id)
print(f"Job ID : {job_id}")
print(f"Status : {status.status}")

recent_events = openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=5).data
for event in recent_events:
    print(f"  {event.message}")

Job ID : ftjob-OuBMK4JJhpALvfHlgKX2tVul
Status : succeeded
  The job has successfully completed
  Usage policy evaluations completed, model is now enabled for sampling
  Moderation checks for snapshot ft:gpt-4.1-nano-2025-04-14:personal:pricer:DGwrI5tw passed.
  Evaluating model against our usage policies
  New fine-tuned model created


In [16]:
# Evaluate base model

def test_messages_for(item):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Estimate the price of this product.\n\n{item.summary}"},
    ]

def base_model_pricer(item):
    response = openai.chat.completions.create(
        model    = BASE_MODEL,
        messages = test_messages_for(item),
        max_tokens = 7
    )
    return response.choices[0].message.content

print("Evaluating base model (no fine-tuning)...")
evaluate(base_model_pricer, test)

Evaluating base model (no fine-tuning)...


  0%|          | 0/200 [00:00<?, ?it/s]

$29 $26 $25 $20 $60 $80 $104 $65 $0 $870 $263 $21 $10 $24 $19 $17 $31 $20 $115 $29 $74 $76 $5 $0 $212 $273 $305 $4 $281 $65 $26 $10 $20 $50 $25 $319 $65 $26 $34 $22 $170 $54 $22 $125 $70 $5 $18 $2 $70 $82 $22 $104 $275 $10 $77 $14 $9 $30 $52 $4 $86 $18 $35 $35 $279 $0 $154 $295 $25 $49 $12 $13 $120 $3 $25 $20 $26 $5 $8 $3 $30 $4 $10 $74 $7 $10 $68 $91 $30 $0 $18 $35 $4 $19 $0 $138 $2 $27 $30 $225 $20 $42 $17 $39 $101 $132 $9 $375 $15 $21 $10 $136 $59 $52 $24 $270 $5 $4 $34 $28 $14 $411 $50 $16 $0 $15 $6 $151 $1 $93 $25 $13 $35 $14 $65 $1 $75 $20 $43 $42 $26 $50 $70 $7 $104 $58 $15 $290 $85 $17 $4 $224 $26 $20 $6 $159 $101 $40 $70 $14 $211 $16 $5 $0 $90 $3 $52 $30 $10 $5 $9 $4 $60 $7 $12 $201 $2 $57 $14 $3 $146 $25 $150 $69 $70 $8 $73 $7 $30 $11 $5 $99 $11 $111 $54 $70 $10 $70 $21 $6 

In [17]:
# Evaluate Fine-Tuned Model
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
print(f"Fine-tuned model: {fine_tuned_model_name}")

def fine_tuned_pricer(item):
    response = openai.chat.completions.create(
        model      = fine_tuned_model_name,
        messages   = test_messages_for(item),
        max_tokens = 7
    )
    return response.choices[0].message.content

print("Evaluating fine-tuned model...")
evaluate(fine_tuned_pricer, test)

Fine-tuned model: ft:gpt-4.1-nano-2025-04-14:personal:pricer:DGwrI5tw
Evaluating fine-tuned model...


  0%|          | 0/200 [00:00<?, ?it/s]

$10 $14 $29 $16 $14 $157 $86 $86 $6 $96 $303 $30 $20 $14 $1 $12 $54 $26 $120 $83 $44 $139 $63 $99 $87 $234 $52 $5 $136 $58 $3 $60 $60 $60 $15 $220 $100 $36 $4 $37 $111 $54 $8 $14 $41 $3 $8 $3 $85 $139 $0 $97 $276 $13 $120 $0 $3 $110 $24 $9 $146 $45 $7 $43 $351 $10 $142 $245 $84 $49 $6 $3 $320 $0 $95 $2 $66 $3 $33 $9 $100 $7 $24 $77 $0 $1 $382 $181 $54 $24 $20 $35 $5 $14 $7 $66 $9 $37 $95 $400 $246 $53 $18 $9 $181 $153 $1 $352 $9 $120 $30 $57 $27 $37 $124 $78 $4 $6 $77 $87 $24 $310 $20 $192 $10 $30 $2 $46 $55 $90 $383 $29 $111 $0 $49 $3 $79 $40 $23 $22 $14 $1 $502 $24 $149 $32 $19 $140 $155 $3 $0 $194 $2 $35 $3 $59 $126 $33 $0 $37 $65 $17 $4 $37 $265 $2 $283 $30 $30 $20 $25 $4 $420 $47 $40 $24 $13 $9 $14 $7 $25 $56 $333 $139 $16 $18 $43 $24 $35 $10 $20 $8 $21 $11 $71 $40 $20 $470 $11 $4 

In [18]:
# Gradio UI
def get_price(text):
    s     = text.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(match.group()) if match else 0.0

def load_product():
    item        = random.choice(test)
    description = item.summary
    return description, "", "", gr.update(interactive=True)

def submit_guess(description, user_guess):
    item       = next(i for i in test if i.summary == description)
    actual     = item.price
    user_price = get_price(user_guess)

    base_response = openai.chat.completions.create(
        model      = BASE_MODEL,
        messages   = test_messages_for(item),
        max_tokens = 7
    ).choices[0].message.content
    base_price = get_price(base_response)

    ft_response = openai.chat.completions.create(
        model      = fine_tuned_model_name,
        messages   = test_messages_for(item),
        max_tokens = 7
    ).choices[0].message.content
    ft_price = get_price(ft_response)

    result = (
        f"Actual price:         ${actual:.2f}\n\n"
        f"Your guess:           ${user_price:.2f}  (error: ${abs(user_price - actual):.2f})\n"
        f"Base model:           ${base_price:.2f}  (error: ${abs(base_price - actual):.2f})\n"
        f"Fine-tuned model:     ${ft_price:.2f}  (error: ${abs(ft_price - actual):.2f})"
    )
    return result

welcome = [{
    "role":    "assistant",
    "content": (
        "🛒 Welcome to the Price is Right!\n\n"
        "A product description will be shown below. Guess its price, then hit Submit "
        "to see how you compare against the base model and the fine-tuned model.\n\n"
        "Hit 'Next Product' to begin!"
    )
}]

with gr.Blocks() as ui:
    gr.Markdown("## 🏷️ The Price is Right — LLM Fine-Tuning Edition")
    gr.Markdown("Guess the product price, then see how you compare against the base and fine-tuned models.")

    chatbot     = gr.Chatbot(value=welcome, height=200, type="messages")
    description = gr.Textbox(label="Product Description", lines=8, interactive=False)
    user_guess  = gr.Textbox(label="Your Price Guess (e.g. 49.99)", placeholder="Enter your guess...")
    result_box  = gr.Textbox(label="Results", lines=6, interactive=False)

    with gr.Row():
        next_btn   = gr.Button("Next Product", variant="secondary")
        submit_btn = gr.Button("Submit Guess", variant="primary")

    next_btn.click(
        fn      = lambda: gr.update(interactive=False),
        outputs = next_btn
    ).then(
        fn      = load_product,
        outputs = [description, user_guess, result_box, submit_btn]
    ).then(
        fn      = lambda: gr.update(interactive=True),
        outputs = next_btn
    )

    submit_btn.click(
        fn      = lambda: gr.update(interactive=False),
        outputs = submit_btn
    ).then(
        fn      = submit_guess,
        inputs  = [description, user_guess],
        outputs = result_box
    ).then(
        fn      = lambda: gr.update(interactive=True),
        outputs = submit_btn
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
